In [1]:
# Select sample size for PM2.5 TMREL and BMR

In [2]:
import os
import xarray as xr
import numpy as np
from utils.utils import create_global_country_map

In [3]:
# === CHOOSE NUMBER OF SAMPLES ===
n_samples = 200

In [5]:
# === Calculate the TMREL distribution ===

# TMREL from GBD21 (uniform distribution)
tmrel_low = 2.4
tmrel_high = 5.9
tmrel_samples = np.random.uniform(tmrel_low, tmrel_high, size=n_samples)

tmrel_da = xr.DataArray(
    tmrel_samples,
    dims=["samples"],
    coords={"samples": np.arange(n_samples)}
).astype("float32")

# === Save file in scratch directory ===
SAVE_DIR = "/glade/derecho/scratch/awells/air_quality/TMREL/"

out_file = f"TMREL_{n_samples}_samples_pm25.nc"
out_path = os.path.join(SAVE_DIR, out_file)
tmrel_da.to_netcdf(out_path)

In [6]:
# === Health variables ===
# COPD, DIABETES, ISCHEMIC_HEART_DISEASE, LOWER_RESPIRATORY_INFECTIONS, LUNG_CANCER, STROKE
# resp_copd, t2_dm, cvd_ihd, lri, neo_lung, cvd_stroke
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER", "STROKE"]

In [7]:
# === Calculate the RR distribution ===

RR_DIR = "/glade/work/awells/air_quality/GBD21/RR_curves/"

for health_VAR in health_vars:
    print(f"Processing {health_VAR}")
    RR_file = f"IHME_GBD_2021_AIR_POLLUTION_1990_2021_PM_RR_{health_VAR}.nc"
    RR_path = os.path.join(RR_DIR, RR_file)
    RR = xr.open_dataset(RR_path)

    # RR from GBD21 curves
    RR_mean = RR["mean"]
    RR_lower = RR["lower"]
    RR_upper = RR["upper"]

    RR_std = (RR_upper - RR_lower) / (2 * 1.96)

    RR_samples = xr.DataArray(
        np.random.normal(
            RR_mean.values[..., np.newaxis],
            RR_std.values[..., np.newaxis],
            size=RR_mean.shape + (n_samples,)
        ),
        dims=RR_mean.dims + ("samples",),
        coords={**RR_mean.coords, "samples": np.arange(n_samples)},
    )

    # === Save file in scratch directory ===
    SAVE_DIR = "/glade/derecho/scratch/awells/air_quality/rr_pm25/"

    out_file = f"RR_{health_VAR}_{n_samples}_samples_pm25.nc"
    out_path = os.path.join(SAVE_DIR, out_file)
    print(f"Saving to {out_path}")
    RR_samples.to_netcdf(out_path)

print("All processing complete.")

Processing COPD
Saving to /glade/derecho/scratch/awells/air_quality/rr_pm25/RR_COPD_200_samples_pm25.nc
Processing DIABETES
Saving to /glade/derecho/scratch/awells/air_quality/rr_pm25/RR_DIABETES_200_samples_pm25.nc
Processing ISCHEMIC_HEART_DISEASE
Saving to /glade/derecho/scratch/awells/air_quality/rr_pm25/RR_ISCHEMIC_HEART_DISEASE_200_samples_pm25.nc
Processing LOWER_RESPIRATORY_INFECTIONS
Saving to /glade/derecho/scratch/awells/air_quality/rr_pm25/RR_LOWER_RESPIRATORY_INFECTIONS_200_samples_pm25.nc
Processing LUNG_CANCER
Saving to /glade/derecho/scratch/awells/air_quality/rr_pm25/RR_LUNG_CANCER_200_samples_pm25.nc
Processing STROKE
Saving to /glade/derecho/scratch/awells/air_quality/rr_pm25/RR_STROKE_200_samples_pm25.nc
All processing complete.


In [8]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"

# Save file in scratch directory ~25GB
SAVE_DIR = "/glade/derecho/scratch/awells/air_quality/BMR/"

In [9]:
# === Calculate the BMR distribution ===

for health_VAR in health_vars:
    print(f"Processing {health_VAR}")

    # Load BMR for each country (lat, lon, quantile)
    bmr_file = f"GBD_BMR_Country_{health_VAR}_newlabels_1990-2009.nc"
    bmr_path = os.path.join(BMR_DIR, bmr_file)
    BMR = xr.open_dataarray(bmr_path)  # three quantiles

    # BMR from vizhub (GBD21)
    bmr_mean = BMR.sel(quantile="mean")
    bmr_lower = BMR.sel(quantile="lower")
    bmr_upper = BMR.sel(quantile="upper")
    bmr_std = (bmr_upper - bmr_lower) / (2 * 1.96)

    bmr_samples = np.random.normal(
        bmr_mean,
        bmr_std,
        size=(n_samples, len(BMR.country)))

    bmr_da = xr.DataArray(
        bmr_samples,
        dims=["samples", "country"],
        coords={"samples": np.arange(n_samples), "country": BMR.country}
    ).astype("float32")

    del bmr_samples

    # WARNING: this step can be slow and use a lot of memory
    # e.g. ~100GB for 1000 samples, ~30GB for 200 samples
    bmr_global = create_global_country_map(bmr_da).chunk({"samples": 10, "lat": 180, "lon": 360})

    # Save file ~25GB
    out_file = f"GBD_BMR_Country_Mask_{health_VAR}_{n_samples}_samples_1990-2009.nc"
    out_path = os.path.join(SAVE_DIR, out_file)
    print(f"Saving to {out_path}")
    bmr_global.to_netcdf(out_path)

    del bmr_global

print("All processing complete.")

Processing COPD
Saving to /glade/derecho/scratch/awells/air_quality/BMR/GBD_BMR_Country_Mask_COPD_200_samples_1990-2009.nc
Processing DIABETES
Saving to /glade/derecho/scratch/awells/air_quality/BMR/GBD_BMR_Country_Mask_DIABETES_200_samples_1990-2009.nc
Processing ISCHEMIC_HEART_DISEASE
Saving to /glade/derecho/scratch/awells/air_quality/BMR/GBD_BMR_Country_Mask_ISCHEMIC_HEART_DISEASE_200_samples_1990-2009.nc
Processing LOWER_RESPIRATORY_INFECTIONS
Saving to /glade/derecho/scratch/awells/air_quality/BMR/GBD_BMR_Country_Mask_LOWER_RESPIRATORY_INFECTIONS_200_samples_1990-2009.nc
Processing LUNG_CANCER
Saving to /glade/derecho/scratch/awells/air_quality/BMR/GBD_BMR_Country_Mask_LUNG_CANCER_200_samples_1990-2009.nc
Processing STROKE
Saving to /glade/derecho/scratch/awells/air_quality/BMR/GBD_BMR_Country_Mask_STROKE_200_samples_1990-2009.nc
All processing complete.
